# The Eight Basis Classes

**Part II · Geometric Algebra** — Tutorial 03

[Tutorial 02](../02_algebra_core/) built algebras with the generic
`Algebra(dim, sig, dtype)` constructor. *pytanga* also ships **eight prebuilt
`Algebra` subclasses** — the *basis classes* — that already know their dimension
and signature and expose every named basis blade as an attribute. A basis class
pre-defines those named blades and is required whenever you use the `Geometry`
submodule (see below).

| 3D class | Algebra | Meaning |
|---|---|---|
| `BasisE3` | G(3, 0) | Euclidean 3D |
| `BasisP3` | G(4, 0) | Projective 3D (homogeneous coordinates) |
| `BasisN3` | G(5, 0b10000) | Null / conformal 3D |
| `BasisPGA3` | G(5, 0b10000) | Plane-based (Gunn/Dorst) PGA 3D |

| 2D class | Algebra | Meaning |
|---|---|---|
| `BasisE2` | G(2, 0) | Euclidean 2D |
| `BasisP2` | G(3, 0) | Projective 2D |
| `BasisN2` | G(4, 0b1000) | Null / conformal 2D |
| `BasisPGA2` | G(4, 0b1000) | Plane-based (Gunn/Dorst) PGA 2D |

> **Convention.** Geometric entities and operators are created through the
> `pytanga.geometry` submodule — not on the basis classes themselves. Every geometric
> example below uses the `Geometry` convenience class (see the
> [Geometry tutorial](../14_geometry/)).


## 1. Setup

All eight classes live in `pytanga.basis`; the geometric entities/operators and the
`Geometry` facade live in `pytanga.geometry`:


In [1]:
from pytanga.basis import (
    BasisE2, BasisE3, BasisP2, BasisP3,
    BasisN2, BasisN3, BasisPGA2, BasisPGA3,
)
from pytanga import MV
from pytanga.geometry import (
    Geometry, Point, Direction, Line, Plane, Sphere, Rotor,
)
import math


## 2. Named blades & the three access patterns

A basis class exposes its named blades as attributes, and a `blades()` helper returns
them all as a `{name: MV}` dict. There are three common ways to get those short names
into your code:


In [2]:
E3 = BasisE3()
print("blades:", list(E3.blades().keys()))

# Pattern 1 — namespace injection (short, but invisible to linters/type-checkers)
globals().update(E3.blades())
print("(e1 * e2)       =", e1 * e2)     # noqa: F821

# Pattern 2 — attribute access (verbose, but fully typed everywhere)
print("(E3.e1 * E3.e2) =", E3.e1 * E3.e2)

# Pattern 3 — explicit assignment block (recommended: short names + full typing)
a1: MV = E3.e1
a2: MV = E3.e2
a3: MV = E3.e3
ps: MV = E3.I     # pseudoscalar, renamed to avoid shadowing 'I'

print("(a1 ^ a2 ^ a3)  =", a1 ^ a2 ^ a3)
print("(ps * ps)       =", ps * ps)      # -1 in G(3,0)


blades: ['e1', 'e2', 'e3', 'e12', 'e13', 'e23', 'e31', 'I']
(e1 * e2)       = e12
(E3.e1 * E3.e2) = e12
(a1 ^ a2 ^ a3)  = I
(ps * ps)       = -1


## 3. The four 3D classes

### `BasisE3` — Euclidean 3D, G(3, 0)

The simplest and most familiar algebra. Its 8 blades are the scalar, the three basis
vectors, the three bivectors, and the pseudoscalar $I = e_1 \wedge e_2 \wedge e_3$.
In G(3,0) the pseudoscalar squares to $-1$.


In [3]:
E3 = BasisE3()     # G(3, 0), float64
print("dim =", E3.dim, "| sig =", E3.sig, "| blades =", E3.algebra_dim)

e1, e2, e3 = E3.e1, E3.e2, E3.e3
I = E3.I

v = E3("e1 + 2 e2 + 3 e3")
print("v        =", v)
print("e1 * e2  =", e1 * e2)      # bivector e12
print("I * I    =", I * I)        # -1 in G(3,0)


dim = 3 | sig = 0 | blades = 8
v        = e1 + 2 e2 + 3 e3
e1 * e2  = e12
I * I    = -1


In [4]:
# E3 represents planes and rotors (not finite points/lines — those need P3/N3/PGA3).
geo = Geometry(BasisE3())

plane = geo(Plane(point=Point(0, 0, 0), normal=Direction(0, 0, 1)))
print("Plane MV  :", plane)
print("analyze   :", geo(plane))

rotor = geo(Rotor(angle=math.pi / 2, axis=Direction(0, 0, 1)))
print("Rotor MV  :", rotor)
print("analyze   :", geo(rotor))


Plane MV  : -e12
analyze   : Plane(pt=Point(0.00, 0.00, 0.00), n=Dir(0.00, 0.00, -1.00))
Rotor MV  : 0.7071 - 0.7071 e12
analyze   : Rotor(90.0° about Dir(0.00, 0.00, 1.00))


### `BasisP3` — Projective 3D, G(4, 0)

Adds a fourth basis vector `e4` (the *homogeneous* coordinate). A finite point
$(x, y, z)$ is the vector $x e_1 + y e_2 + z e_3 + e_4$; leaving out `e4` gives an
ideal point (a direction at infinity).


In [5]:
P3 = BasisP3()
print("dim =", P3.dim, "| blades =", list(P3.blades().keys()))

# Homogeneous point via string: x·e1 + y·e2 + z·e3 + e4
p = P3("e1 + 2 e2 + 3 e3 + e4")
print("homogeneous point:", p)

geo = Geometry(P3)
mv = geo(Point(1, 2, 3))
print("Point MV  :", mv)
print("analyze   :", geo(mv))

line = geo(Line(origin=Point(0, 0, 0), direction=Direction(1, 0, 0)))
print("Line MV   :", line)
print("analyze   :", geo(line))


dim = 4 | blades = ['e1', 'e2', 'e3', 'e4', 'e12', 'e13', 'e23', 'e31', 'e123', 'I']
homogeneous point: e1 + 2 e2 + 3 e3 + e4
Point MV  : e1 + 2 e2 + 3 e3 + e4
analyze   : Point(1.00, 2.00, 3.00)
Line MV   : -e14
analyze   : Line(org=Point(-0.00, -0.00, -0.00), dir=Dir(-1.00, 0.00, 0.00))


### `BasisN3` — Null / conformal 3D, G(5, 0b10000)

Adds two extra basis vectors `ep` ($e_p^2 = +1$) and `em` ($e_m^2 = -1$), which are
combined into the two *null* vectors

$$\mathrm{einf} = e_p + e_m, \qquad e_o = \tfrac{1}{2}e_m - \tfrac{1}{2}e_p .$$

These satisfy $\mathrm{einf}^2 = 0$, $e_o^2 = 0$, and $e_o \cdot \mathrm{einf} = -1$,
and make spheres, circles, and point pairs first-class entities.


In [6]:
N3 = BasisN3()
print("dim =", N3.dim, "| blades =", list(N3.blades().keys()))

einf, eo = N3.einf, N3.eo
print("einf      =", einf)
print("eo        =", eo)
print("einf^2    =", einf * einf)      # null
print("eo^2      =", eo * eo)          # null
print("eo·einf   =", N3.ip(eo, einf))  # -1

geo = Geometry(N3)
mv = geo(Point(1, 2, 3))
print("Point MV  :", mv)
print("analyze   :", geo(mv))

sph = geo(Sphere(center=Point(1, 2, 3), radius=5.0))
print("Sphere MV :", sph)
print("analyze   :", geo(sph))


dim = 5 | blades = ['e1', 'e2', 'e3', 'e12', 'e13', 'e23', 'e31', 'ep', 'em', 'einf', 'eo', 'I', 'E']
einf      = einf
eo        = eo
einf^2    = 0
eo^2      = 0
eo·einf   = -1
Point MV  : e1 + 2 e2 + 3 e3 + 7 einf + eo
analyze   : Point(1.00, 2.00, 3.00)
Sphere MV : -5.5 e123∧einf - e123∧eo - 3 e12∧einf∧eo + 2 e13∧einf∧eo - e23∧einf∧eo
analyze   : Sphere(c=Point(1.00, 2.00, 3.00), r=5.00)


### `BasisPGA3` — Plane-based (Gunn/Dorst) PGA 3D

Uses the Gunn/Dorst convention: a single null vector `e0` ($e_0^2 = 0$) with a
reciprocal `e0_recip` (satisfying $\langle e_0 \cdot e_0^{\mathrm{recip}} \rangle_0 = 1$). In
plane-based PGA, *planes* are the grade-1 primitives, and points are the grade-3
trivectors. A finite point carries `e0`; an ideal point (a direction) does not.


In [7]:
PGA = BasisPGA3()
print("dim =", PGA.dim, "| blades =", list(PGA.blades().keys()))

e0 = PGA.e0
print("e0 * e0     =", e0 * e0)             # null
print("e0 * e0_recip =", e0 * PGA.e0_recip)     # 1 + E

finite = PGA("e1 + 2 e2 + 3 e3 + e0")   # finite point
ideal  = PGA("e1 + 2 e2 + 3 e3")        # ideal point / direction (no e0)
print("finite point:", finite)
print("ideal point :", ideal)

geo = Geometry(PGA)
mv = geo(Point(1, 2, 3))
print("Point MV  :", mv)
print("analyze   :", geo(mv))

line = geo(Line(origin=Point(0, 0, 0), direction=Direction(1, 0, 0)))
print("Line MV   :", line)
print("analyze   :", geo(line))


dim =

 5 | blades = ['e1', 'e2', 'e3', 'e12', 'e13', 'e23', 'e31', 'ep', 'em', 'e0', 'e0_recip']
e0 * e0     = 0
e0 * e0_recip = 1 + E
finite point: e0 + e1 + 2 e2 + 3 e3
ideal point : e1 + 2 e2 + 3 e3
Point MV  : e032 + 2 e013 + 3 e021 + e123
analyze   : Point(1.00, 2.00, 3.00)
Line MV   : e23
analyze   : Line(org=Point(0.00, 0.00, 0.00), dir=Dir(1.00, 0.00, -0.00))


## 4. The four 2D classes

The 2D classes use the **same entity and operator dataclasses** as 3D — the `z`
component is simply always `0`. Two things to remember:

- **`BasisE2` has no points** — only directions and rotors.
- In **`BasisN2` a "sphere" is a circle** (the conformal model needs 3 points to
  define a sphere, which in 2D collapses to a circle).

### `BasisE2` — Euclidean 2D, G(2, 0)  ·  `BasisP2` — Projective 2D, G(3, 0)


In [8]:
E2 = BasisE2()
print("E2: dim =", E2.dim, "| blades =", list(E2.blades().keys()))

v = E2("3 e1 + 4 e2")           # directions only — no points in E2
print("vector  :", v)
print("I * I   :", E2.I * E2.I)  # -1 in G(2,0)

P2 = BasisP2()
print("P2: dim =", P2.dim, "| blades =", list(P2.blades().keys()))

geo = Geometry(P2)
mv = geo(Point(3, 4, 0))   # homogeneous point; z is always 0
print("Point MV:", mv)
print("analyze :", geo(mv))


E2: dim = 2 | blades = ['e1', 'e2', 'e12', 'I']
vector  : 3 e1 + 4 e2
I * I   : -1


P2: dim = 3 | blades = ['e1', 'e2', 'e3', 'e123', 'I']
Point MV: 3 e1 + 4 e2 + e3
analyze : Point(3.00, 4.00, 0.00)


### `BasisN2` — Null/conformal 2D, G(4, 0b1000)  ·  `BasisPGA2` — PGA 2D

`BasisN2` mirrors `BasisN3` with one fewer Euclidean dimension (`ep = e3`, `em = e4`),
so it exposes `einf` and `eo`. `BasisPGA2` uses the Gunn/Dorst `e0` / `e0_recip`
convention, where lines are the grade-1 primitives and points are grade-2 bivectors.


In [9]:
N2 = BasisN2()
print("N2: dim =", N2.dim, "| blades =", list(N2.blades().keys()))

geo = Geometry(N2)
mv = geo(Point(3, 4, 0))
print("Point MV :", mv)
print("analyze  :", geo(mv))

# In N2 a "Sphere" is a circle (3 points define a sphere -> a circle in 2D):
sph = geo(Sphere(center=Point(0, 0, 0), radius=2.0))
print("Sphere MV:", sph)
print("analyze  :", geo(sph))

PGA2 = BasisPGA2()
print("PGA2: dim =", PGA2.dim, "| blades =", list(PGA2.blades().keys()))

geo2 = Geometry(PGA2)
mv2 = geo2.create(Point(3, 4, 0))
print("Point MV :", mv2)
print("analyze  :", geo2.analyze(mv2))


N2: dim = 4 | blades = ['e1', 'e2', 'ep', 'em', 'einf', 'eo', 'I', 'E']
Point MV : 3 e1 + 4 e2 + 12.5 einf + eo
analyze  : Point(3.00, 4.00, 0.00)
Sphere MV: 2 e12∧einf + e12∧eo
analyze  : Circle(c=Point(0.00, 0.00, 0.00), r=2.00, n=Dir(0.00, 0.00, 1.00))
PGA2: dim = 4 | blades = ['e1', 'e2', 'ep', 'em', 'e0', 'e0_recip']
Point MV : 4 e10 - 3 e20 - e12
analyze  : Point(3.00, 4.00, 0.00)


## 5. Side-by-side comparison

The eight classes differ only in dimension, signature, and the set of named blades
they expose:


In [10]:
classes = [
    ("E3", BasisE3), ("P3", BasisP3), ("N3", BasisN3), ("PGA3", BasisPGA3),
    ("E2", BasisE2), ("P2", BasisP2), ("N2", BasisN2), ("PGA2", BasisPGA2),
]

for name, B in classes:
    b = B()
    print(f"{name:>5}: G({b.dim}, {b.sig:#05b})  dtype={b.dtype}  blades={b.algebra_dim}")
    print(f"        named blades: {', '.join(b.blades())}")


   E3: G(3, 0b000)  dtype=float64  blades=8
        named blades: e1, e2, e3, e12, e13, e23, e31, I
   P3: G(4, 0b000)  dtype=float64  blades=16
        named blades: e1, e2, e3, e4, e12, e13, e23, e31, e123, I
   N3: G(5, 0b10000)  dtype=float64  blades=32
        named blades: e1, e2, e3, e12, e13, e23, e31, ep, em, einf, eo, I, E
 PGA3: G(5, 0b10000)  dtype=float64  blades=32
        named blades: e1, e2, e3, e12, e13, e23, e31, ep, em, e0, e0_recip
   E2: G(2, 0b000)  dtype=float64  blades=4
        named blades: e1, e2, e12, I


   P2: G(3, 0b000)  dtype=float64  blades=8
        named blades: e1, e2, e3, e123, I
   N2: G(4, 0b1000)  dtype=float64  blades=16
        named blades: e1, e2, ep, em, einf, eo, I, E
 PGA2: G(4, 0b1000)  dtype=float64  blades=16
        named blades: e1, e2, ep, em, e0, e0_recip


## 6. Entity & operator coverage

Not every algebra can represent every entity. These matrices (taken from the geometry
docs) summarize what `Geometry.create()` / `geo()` support.

### Entities

| Entity | E3 | P3 | PGA3 | N3 | E2 | P2 | PGA2 | N2 |
|--------|:--:|:--:|:----:|:--:|:--:|:--:|:----:|:--:|
| Point | — | ✓ | ✓ | ✓ | — | ✓ | ✓ | ✓ |
| Direction | — | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |
| PointPair | — | — | — | ✓ | — | — | — | ✓ |
| ImagPointPair | — | — | — | ✓ | — | — | — | ✓ |
| Line | — | ✓ | ✓ | ✓ | — | ✓ | ✓ | ✓ |
| Circle | — | — | — | ✓ | — | — | — | ✓ |
| ImagCircle | — | — | — | ✓ | — | — | — | ✓ |
| Plane | ✓ | ✓ | ✓ | ✓ | — | — | ✓ | — |
| Sphere | — | — | — | ✓ | — | — | — | ✓ |
| ImagSphere | — | — | — | ✓ | — | — | — | ✓ |
| Space | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |

### Operators

| Operator | E3 | P3 | PGA3 | N3 | E2 | P2 | PGA2 | N2 |
|----------|:--:|:--:|:----:|:--:|:--:|:--:|:----:|:--:|
| ReflectionPlane | ✓ | ✓ | ✓ | ✓ | — | ✓ | ✓ | ✓ |
| ReflectionLine | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |
| ReflectionPoint | — | ✓ | ✓ | ✓ | — | ✓ | ✓ | ✓ |
| Inversion | — | — | — | ✓ | — | — | — | ✓ |
| Rotor | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |
| Translator | — | — | ✓ | ✓ | — | — | — | ✓ |
| Dilator | — | — | — | ✓ | — | — | — | ✓ |
| Motor | — | — | ✓ | ✓ | — | — | — | ✓ |
| GeneralRotor | — | — | ✓ | ✓ | — | — | — | ✓ |


## 7. Visual examples

To make the comparison concrete, here is the *same* geometric point `(1, 2, 0)` created
in three different algebras — note the three different multivector forms:


In [11]:
for name, B in [("P3", BasisP3), ("PGA3", BasisPGA3), ("N3", BasisN3)]:
    mv = Geometry(B()).create(Point(1, 2, 0))
    print(f"{name:>4}: {mv}")


  P3: e1 + 2 e2 + e4
PGA3: e032 + 2 e013 + e123
  N3: e1 + 2 e2 + 2.5 einf + eo


The `pytanga.viz` viewer renders each multivector as the *same kind* of object,
regardless of which algebra produced it. (For the full viewer API see
[Part I — Visualization](../../visualization/).) `display_snapshot()` renders the scene
**inline** — no server or exported file needed:


In [12]:
from pytanga.viz import Visualizer

viz = Visualizer(title="Entities across P3, PGA3, N3", space_dim=3)
viz.add(Geometry(BasisP3()).create(Point(1, 2, 0)), color="#ff4444", label="P3 point")
viz.add(Geometry(BasisPGA3()).create(Point(2, -1, 0)), color="#44ff44", label="PGA3 point")
viz.add(Geometry(BasisN3()).create(Point(-1, 1, 0)), color="#4488ff", label="N3 point")
viz.add(Geometry(BasisP3()).create(Line(origin=Point(-1, -1, 0), direction=Direction(2, 3, 0))),
        color="#ffaa00", label="P3 line")

viz.display_snapshot()   # renders the scene inline (serverless)


And the 2D counterparts, again rendered inline with `Visualizer(space_dim=2)`:


In [13]:
viz2 = Visualizer(title="2D counterparts (P2, N2, PGA2)", space_dim=2)
viz2.add(Geometry(BasisP2()).create(Point(3, 4, 0)), color="#ff4444", label="P2")
viz2.add(Geometry(BasisN2()).create(Point(-2, 2, 0)), color="#44ff44", label="N2")
viz2.add(Geometry(BasisPGA2()).create(Point(1, -3, 0)), color="#4488ff", label="PGA2")

viz2.display_snapshot()   # renders the scene inline (serverless)


## 8. Summary & next steps

You now know the eight basis classes, their signatures and named blades, the three
patterns for accessing blades, and which entities/operators each algebra supports.

**Where to go next** — a deep dive into each 3D algebra:

- [**04 · Euclidean 3D**](../04_euclidean_e3/) — vectors, bivectors, rotors in `BasisE3`
- [**05 · Projective 3D**](../05_projective_p3/) — homogeneous coordinates in `BasisP3`
- [**06 · Conformal 3D**](../06_conformal_n3/) — spheres, circles, point pairs in `BasisN3`
- [**07 · PGA 3D**](../07_pga3/) — plane-based PGA in `BasisPGA3`
- [**14 · Geometry Submodule**](../14_geometry/) — the `Geometry` facade and
  entity/operator pipeline used throughout this tutorial

> The generic `Algebra` constructor from [Tutorial 02](../02_algebra_core/) remains
> available for custom dimensions and signatures; the basis classes pre-define the
> named blades and are required by the `Geometry` submodule for the eight standard
> geometries.